# S5 — Combined Visualization & Analysis

Combines S3 signal data with S4's True Null + Variance-only calibration data.

**Data sources:**
- S3: 114,176 cases (32 True Null + 114,048 Signal) → `full/S3/_perm_phase1.npz`, `_perm_phase2.npz`
- S4: 6,880 cases → `full/S4_null_validate_all/_core_arrays.npz`, `null_meta.csv`
  - True Null (constant spread): 4,000 cases — for FPR validation
  - Variance-only (non-constant spread): 2,880 cases — for detection power analysis

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy import stats

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 150,
                     'figure.facecolor': 'white', 'font.size': 10})

DATA_DIR = Path('generated_scatterplot_data')
S3_DIR   = DATA_DIR / 'full' / 'S3'
S4_DIR   = DATA_DIR / 'full' / 'S4_null_validate_all'
VIZ_DIR  = DATA_DIR / 'full' / 'S5_viz'
VIZ_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print(f'Output → {VIZ_DIR}')

## 1. Load Data (S3 + S4)

In [2]:
# --- S3 data ---
p1 = np.load(S3_DIR / '_perm_phase1.npz')
p2 = np.load(S3_DIR / '_perm_phase2.npz')

s3_pearson_obs  = p1['pearson_obs']
s3_spearman_obs = p1['spearman_obs']
s3_eta2_obs     = p1['eta2_obs']
s3_pearson_null  = p1['pearson_null']
s3_spearman_null = p1['spearman_null']
s3_eta2_null     = p1['eta2_null']
s3_dcor_obs  = p2['dcor_obs']
s3_dcor_null = p2['dcor_null']

cases = pd.read_csv(DATA_DIR / 'cases.csv', low_memory=False)
n_s3 = len(s3_pearson_obs)
n_perm = s3_pearson_null.shape[1]

# Identify S3 True Null (constant noise)
s3_null_mask = (cases['family_id'] == 'Null') & (cases['spread_pattern'] == 'constant')
s3_null_idx = np.where(s3_null_mask)[0]
s3_signal_mask = cases['family_id'] != 'Null'

print(f'S3: {n_s3:,} cases, {n_perm} permutations')
print(f'  True Null (constant): {len(s3_null_idx)}')
print(f'  Signal: {s3_signal_mask.sum():,}')

S3: 114,176 cases, 500 permutations
  True Null (constant): 32
  Signal: 114,048


In [ ]:
# --- S4 data (new format: _core_arrays.npz + null_meta.csv) ---
s4_core = np.load(S4_DIR / '_core_arrays.npz')

s4_pearson_obs   = s4_core['pearson_obs']
s4_spearman_obs  = s4_core['spearman_obs']
s4_eta2_obs      = s4_core['eta2_obs']
s4_dcor_obs      = s4_core['dcor_obs']
s4_pearson_null  = s4_core['pearson_null']
s4_spearman_null = s4_core['spearman_null']
s4_eta2_null     = s4_core['eta2_null']
s4_dcor_null     = s4_core['dcor_null']
s4_dcov_obs      = s4_core['dcov_obs']
s4_dcov_null     = s4_core['dcov_null']

s4_meta = pd.read_csv(S4_DIR / 'null_meta.csv')
n_s4 = len(s4_pearson_obs)

s4_true_null_mask = s4_meta['category'] == 'true_null'
s4_var_only_mask  = s4_meta['category'] == 'variance_only'
s4_null_idx = np.where(s4_true_null_mask)[0]
s4_var_idx  = np.where(s4_var_only_mask)[0]

print(f'S4: {n_s4:,} total cases')
print(f'  True Null:      {len(s4_null_idx):,}')
print(f'  Variance-only:  {len(s4_var_idx):,}')
print(f'  Spread patterns: {s4_meta["spread_pattern"].value_counts().to_dict()}')

In [ ]:
# Combine True Null only: S3's 32 + S4's True Null (constant spread)
null_pearson_obs  = np.concatenate([s3_pearson_obs[s3_null_idx],  s4_pearson_obs[s4_null_idx]])
null_spearman_obs = np.concatenate([s3_spearman_obs[s3_null_idx], s4_spearman_obs[s4_null_idx]])
null_dcor_obs     = np.concatenate([s3_dcor_obs[s3_null_idx],     s4_dcor_obs[s4_null_idx]])
null_eta2_obs     = np.concatenate([s3_eta2_obs[s3_null_idx],     s4_eta2_obs[s4_null_idx]])

null_pearson_null  = np.concatenate([s3_pearson_null[s3_null_idx],  s4_pearson_null[s4_null_idx]])
null_spearman_null = np.concatenate([s3_spearman_null[s3_null_idx], s4_spearman_null[s4_null_idx]])
null_dcor_null     = np.concatenate([s3_dcor_null[s3_null_idx],     s4_dcor_null[s4_null_idx]])
null_eta2_null     = np.concatenate([s3_eta2_null[s3_null_idx],     s4_eta2_null[s4_null_idx]])

# Combined metadata
s3_null_xdist = cases.loc[s3_null_mask, 'x_distribution'].values
s4_null_xdist = s4_meta.loc[s4_true_null_mask, 'x_distribution'].values
null_xdist = np.concatenate([s3_null_xdist, s4_null_xdist])
null_source = np.array(['S3']*len(s3_null_idx) + ['S4']*len(s4_null_idx))

n_null_total = len(null_pearson_obs)
print(f'Combined True Null: {n_null_total} (S3: {len(s3_null_idx)}, S4: {len(s4_null_idx)})')

# Variance-only arrays (for Section 11)
var_pearson_obs  = s4_pearson_obs[s4_var_idx]
var_spearman_obs = s4_spearman_obs[s4_var_idx]
var_dcor_obs     = s4_dcor_obs[s4_var_idx]
var_eta2_obs     = s4_eta2_obs[s4_var_idx]
var_dcov_obs     = s4_dcov_obs[s4_var_idx]

var_pearson_null  = s4_pearson_null[s4_var_idx]
var_spearman_null = s4_spearman_null[s4_var_idx]
var_dcor_null     = s4_dcor_null[s4_var_idx]
var_eta2_null     = s4_eta2_null[s4_var_idx]
var_dcov_null     = s4_dcov_null[s4_var_idx]

var_meta = s4_meta.loc[s4_var_only_mask].reset_index(drop=True)
n_var = len(var_pearson_obs)
print(f'Variance-only: {n_var} cases')

## 2. Null Distribution Histograms (Selected Cases)

Show permutation null histograms for representative cases:
True Null (S4), signal strong, signal weak, nonlinear.

In [ ]:
# Select representative cases
# S4 True Null cases: pick one per x_distribution type
s4_null_meta = s4_meta.loc[s4_true_null_mask].reset_index(drop=False)
s4_selected = {}
for xd in ['even', 'clusters_2', 'center_dense']:
    row = s4_null_meta[s4_null_meta['x_distribution'] == xd].iloc[0]
    s4_selected[f'True Null ({xd})'] = ('s4', int(row['index']))

# S4 Variance-only cases: pick one per spread_pattern
s4_var_meta = s4_meta.loc[s4_var_only_mask].reset_index(drop=False)
for sp in ['increasing', 'middle_high']:
    row = s4_var_meta[(s4_var_meta['spread_pattern'] == sp) &
                      (s4_var_meta['x_distribution'] == 'even')].iloc[0]
    s4_selected[f'Var-only ({sp})'] = ('s4', int(row['index']))

# S3 signal cases
sig_strong = cases[(cases['family_id'] == 'F01') & (cases['snr'] == np.inf) &
                   (cases['spread_pattern'] == 'constant')].index
sig_weak   = cases[(cases['family_id'] == 'F20') & (cases['snr'] == 0.1) &
                   (cases['spread_pattern'] == 'constant')].index
sig_nonlin = cases[(cases['family_id'] == 'F09') & (cases['snr'] == 1.0) &
                   (cases['spread_pattern'] == 'constant')].index

s3_selected = {}
if len(sig_strong) > 0: s3_selected['F01 linear SNR=∞']  = ('s3', sig_strong[0])
if len(sig_weak) > 0:   s3_selected['F20 weak SNR=0.1']  = ('s3', sig_weak[0])
if len(sig_nonlin) > 0: s3_selected['F09 nonlin SNR=1']  = ('s3', sig_nonlin[0])

selected = {**s4_selected, **s3_selected}
print(f'Selected {len(selected)} cases for visualization')

In [6]:
def _get_arrays(source, idx):
    if source == 's3':
        return [(s3_pearson_obs[idx], s3_pearson_null[idx]),
                (s3_spearman_obs[idx], s3_spearman_null[idx]),
                (s3_dcor_obs[idx], s3_dcor_null[idx]),
                (s3_eta2_obs[idx], s3_eta2_null[idx])]
    else:
        return [(s4_pearson_obs[idx], s4_pearson_null[idx]),
                (s4_spearman_obs[idx], s4_spearman_null[idx]),
                (s4_dcor_obs[idx], s4_dcor_null[idx]),
                (s4_eta2_obs[idx], s4_eta2_null[idx])]

metric_names = ['|Pearson|', '|Spearman|', 'dcor', 'η²']

fig, axes = plt.subplots(len(selected), 4, figsize=(18, 3*len(selected)))
fig.suptitle('Null Distributions (500 permutations) with Observed Value', fontsize=14, y=1.01)

for row, (label, (src, idx)) in enumerate(selected.items()):
    arrays = _get_arrays(src, idx)
    for col, (mname, (obs_val, null_vals)) in enumerate(zip(metric_names, arrays)):
        ax = axes[row, col]
        nv = null_vals.astype(np.float64)
        ov = float(obs_val)
        ax.hist(nv, bins=40, color='steelblue', alpha=0.7, edgecolor='white', density=True)
        ax.axvline(ov, color='red', lw=2, label=f'obs={ov:.3f}')
        med = np.median(nv)
        ax.axvline(med, color='orange', lw=1, ls='--', label=f'med={med:.3f}')
        p_perm = (np.sum(nv >= ov) + 1) / (n_perm + 1)
        ax.set_title(f'{mname}  p={p_perm:.3f}', fontsize=9)
        ax.legend(fontsize=7, loc='upper right')
        if col == 0:
            ax.set_ylabel(label, fontsize=9)

plt.tight_layout()
fig.savefig(VIZ_DIR / '1_null_distributions_selected.png', bbox_inches='tight')
plt.show()
print(f'Saved → {VIZ_DIR}/1_null_distributions_selected.png')

Saved → generated_scatterplot_data/full/S5_viz/1_null_distributions_selected.png


/var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/ipykernel_67582/2875296185.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Null Distribution Shape Analysis

Skewness, kurtosis, and normality of the 500-permutation null distributions.
Uses S3 signal cases for general shape + combined True Null for calibration.

In [7]:
# Sample 2000 cases from S3 for general shape analysis
sample_idx = rng.choice(n_s3, size=min(2000, n_s3), replace=False)

shape_records = []
for idx in sample_idx:
    for mname, null_arr in [('pearson', s3_pearson_null), ('spearman', s3_spearman_null),
                             ('dcor', s3_dcor_null), ('eta2', s3_eta2_null)]:
        vals = null_arr[idx].astype(np.float64)
        skew = float(stats.skew(vals))
        kurt = float(stats.kurtosis(vals))
        _, sw_p = stats.shapiro(vals[:50])
        shape_records.append({
            'idx': idx, 'metric': mname,
            'skewness': skew, 'kurtosis': kurt, 'shapiro_p': sw_p,
        })

shape_df = pd.DataFrame(shape_records)
print('Null distribution shape (sampled 2000 S3 cases × 4 metrics):')
print(shape_df.groupby('metric')[['skewness','kurtosis','shapiro_p']].describe()
      .round(3).to_string())

Null distribution shape (sampled 2000 S3 cases × 4 metrics):
         skewness                                                  kurtosis                                                   shapiro_p                                               
            count   mean    std    min    25%    50%    75%    max    count   mean    std    min    25%    50%    75%     max     count   mean    std  min    25%    50%    75%    max
metric                                                                                                                                                                                
dcor       2000.0  1.219  0.202  0.598  1.082  1.203  1.341  2.340   2000.0  2.123  1.097  0.092  1.379  1.940  2.650  13.021    2000.0  0.036  0.102  0.0  0.000  0.002  0.019  0.997
eta2       2000.0  1.022  0.387  0.471  0.835  0.954  1.100  8.284   2000.0  1.766  3.199 -0.173  0.754  1.219  1.895  87.258    2000.0  0.082  0.152  0.0  0.002  0.016  0.084  0.990
pearson    2000.0  0.973

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for mname, color in [('pearson','#1f77b4'), ('spearman','#ff7f0e'),
                      ('dcor','#2ca02c'), ('eta2','#d62728')]:
    sub = shape_df[shape_df['metric'] == mname]
    axes[0].hist(sub['skewness'], bins=50, alpha=0.5, label=mname, density=True)
    axes[1].hist(sub['kurtosis'], bins=50, alpha=0.5, label=mname, density=True)
    axes[2].hist(np.log10(sub['shapiro_p'].clip(1e-30)), bins=50, alpha=0.5, label=mname, density=True)

axes[0].set_title('Skewness of null distributions')
axes[0].axvline(0, color='k', ls='--', lw=0.8)
axes[1].set_title('Excess kurtosis')
axes[1].axvline(0, color='k', ls='--', lw=0.8)
axes[2].set_title('Shapiro-Wilk p-value (log₁₀)')
axes[2].axvline(np.log10(0.05), color='k', ls='--', lw=0.8, label='p=0.05')

for ax in axes: ax.legend(fontsize=8)
plt.suptitle('Shape of Null Distributions (n=500 permutations each)', fontsize=13)
plt.tight_layout()
fig.savefig(VIZ_DIR / '2_null_shape_analysis.png', bbox_inches='tight')
plt.show()
print(f'Saved → {VIZ_DIR}/2_null_shape_analysis.png')

Saved → generated_scatterplot_data/full/S5_viz/2_null_shape_analysis.png


/var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/ipykernel_67582/3509351693.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Null Distribution Properties by x_distribution and spread_pattern

How do null median and IQR vary with case parameters? (S3 data)

In [9]:
# Compute null median and IQR for all S3 cases
null_props = pd.DataFrame({'case_idx': np.arange(n_s3)})

for mname, null_arr in [('pearson', s3_pearson_null), ('spearman', s3_spearman_null),
                         ('dcor', s3_dcor_null), ('eta2', s3_eta2_null)]:
    null_f64 = null_arr.astype(np.float64)
    null_props[f'{mname}_null_med'] = np.median(null_f64, axis=1)
    q75 = np.percentile(null_f64, 75, axis=1)
    q25 = np.percentile(null_f64, 25, axis=1)
    null_props[f'{mname}_null_iqr'] = q75 - q25

null_props['family_id'] = cases['family_id'].values
null_props['snr'] = cases['snr'].values
null_props['spread_pattern'] = cases['spread_pattern'].values
null_props['x_distribution'] = cases['x_distribution'].values

print(null_props.describe().round(4).to_string())

          case_idx  pearson_null_med  pearson_null_iqr  spearman_null_med  spearman_null_iqr  dcor_null_med  dcor_null_iqr  eta2_null_med  eta2_null_iqr       snr
count  114176.0000       114176.0000       114176.0000        114176.0000        114176.0000    114176.0000    114176.0000    114176.0000    114176.0000  114176.0
mean    57087.5000            0.0303            0.0371             0.0302             0.0371         0.0665         0.0209         0.0153         0.0104       inf
std     32959.9165            0.0016            0.0020             0.0016             0.0021         0.0074         0.0020         0.0018         0.0009       NaN
min         0.0000            0.0000            0.0000             0.0000             0.0000         0.0000         0.0000         0.0000         0.0000       0.1
25%     28543.7500            0.0292            0.0358             0.0292             0.0357         0.0621         0.0195         0.0144         0.0098       1.0
50%     57087.5000    

/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [10]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Null Distribution Properties by x_distribution (S3)', fontsize=13)

for col, mname in enumerate(['pearson', 'spearman', 'dcor', 'eta2']):
    for xd in sorted(null_props['x_distribution'].unique()):
        sub = null_props[null_props['x_distribution'] == xd]
        axes[0, col].hist(sub[f'{mname}_null_med'], bins=50, alpha=0.4, label=xd, density=True)
        axes[1, col].hist(sub[f'{mname}_null_iqr'], bins=50, alpha=0.4, label=xd, density=True)
    axes[0, col].set_title(f'{mname} — null median')
    axes[1, col].set_title(f'{mname} — null IQR')
    axes[0, col].legend(fontsize=6)

plt.tight_layout()
fig.savefig(VIZ_DIR / '3_null_props_by_xdist.png', bbox_inches='tight')
plt.show()

/var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/ipykernel_67582/690426402.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Null Distribution Properties by spread_pattern (S3)', fontsize=13)
colors_sp = {'constant': '#2ca02c', 'increasing': '#d62728',
             'decreasing': '#1f77b4', 'middle_high': '#ff7f0e'}

for col, mname in enumerate(['pearson', 'spearman', 'dcor', 'eta2']):
    for sp, c in colors_sp.items():
        sub = null_props[null_props['spread_pattern'] == sp]
        axes[0, col].hist(sub[f'{mname}_null_med'], bins=50, alpha=0.4, label=sp, color=c, density=True)
        axes[1, col].hist(sub[f'{mname}_null_iqr'], bins=50, alpha=0.4, label=sp, color=c, density=True)
    axes[0, col].set_title(f'{mname} — null median')
    axes[1, col].set_title(f'{mname} — null IQR')
    axes[0, col].legend(fontsize=7)

plt.tight_layout()
fig.savefig(VIZ_DIR / '4_null_props_by_spread.png', bbox_inches='tight')
plt.show()

/var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/ipykernel_67582/3380674367.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Z-Score Calibration Under True Null (Combined S3+S4)

**Key analysis**: with 4,032 True Null cases, check whether the
Z-score normalization is well-calibrated under the null.

In [12]:
# Z-score calibration using ALL True Null cases (S3 + S4 combined)
null_metric_info = [
    ('|Pearson|',  null_pearson_obs,  null_pearson_null),
    ('|Spearman|', null_spearman_obs, null_spearman_null),
    ('dcor',       null_dcor_obs,     null_dcor_null),
    ('η²',        null_eta2_obs,     null_eta2_null),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle(f'Z-score Distribution Under True Null (n={n_null_total} cases × 500 perms)', fontsize=13)

for col, (mname, obs_arr, null_arr) in enumerate(null_metric_info):
    all_z = []
    for i in range(n_null_total):
        nv = null_arr[i].astype(np.float64)
        med = np.median(nv)
        iqr = np.percentile(nv, 75) - np.percentile(nv, 25)
        if iqr < 1e-12:
            iqr = np.std(nv) * 1.35
        if iqr > 1e-12:
            z = (nv - med) / iqr
            all_z.extend(z.tolist())
    all_z = np.array(all_z)
    ax = axes[col]
    ax.hist(all_z, bins=80, density=True, alpha=0.7, color='steelblue')
    xr = np.linspace(-4, 6, 200)
    ax.plot(xr, stats.norm.pdf(xr), 'k--', lw=1, label='N(0,1)')
    ax.set_title(f'{mname}\nmed={np.median(all_z):.2f}, IQR={np.percentile(all_z,75)-np.percentile(all_z,25):.2f}')
    ax.set_xlim(-5, 8)
    ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(VIZ_DIR / '5_zscore_calibration_combined.png', bbox_inches='tight')
plt.show()
print(f'Saved → {VIZ_DIR}/5_zscore_calibration_combined.png')

Saved → generated_scatterplot_data/full/S5_viz/5_zscore_calibration_combined.png


/var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/ipykernel_67582/36026633.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. FP Deep Dive (Combined True Null)

Which metric drives false positives? How do FP cases differ from TN cases?

In [13]:
# Compute per-case joint test results for combined True Null
fp_records = []
for i in range(n_null_total):
    z_list, z_null_list = [], []
    for mname, obs_arr, null_arr in [('pearson', null_pearson_obs, null_pearson_null),
                                     ('spearman', null_spearman_obs, null_spearman_null),
                                     ('dcor', null_dcor_obs, null_dcor_null),
                                     ('eta2', null_eta2_obs, null_eta2_null)]:
        obs = float(obs_arr[i])
        nv = null_arr[i].astype(np.float64)
        med = np.median(nv)
        iqr = np.percentile(nv, 75) - np.percentile(nv, 25)
        if iqr < 1e-12: iqr = np.std(nv) * 1.35
        if iqr < 1e-12:
            z_o, z_n = 0.0, np.zeros(n_perm)
        else:
            z_o = (obs - med) / iqr
            z_n = (nv - med) / iqr
        z_list.append(z_o)
        z_null_list.append(z_n)

    T_obs = max(z_list)
    T_null = np.stack(z_null_list).max(axis=0)
    p_val = (np.sum(T_null >= T_obs) + 1) / (n_perm + 1)
    driver = ['pearson','spearman','dcor','eta2'][int(np.argmax(z_list))]

    fp_records.append({
        'x_distribution': null_xdist[i], 'source': null_source[i],
        'T_joint': T_obs, 'p_value': p_val, 'driver': driver,
        'z_pearson': z_list[0], 'z_spearman': z_list[1],
        'z_dcor': z_list[2], 'z_eta2': z_list[3],
        'is_fp': p_val <= 0.05,
    })

fp_df = pd.DataFrame(fp_records)
n_fp = fp_df['is_fp'].sum()
print(f'Combined True Null: {n_null_total} cases, {n_fp} FP ({n_fp/n_null_total:.1%})')

Combined True Null: 4032 cases, 217 FP (5.4%)


In [14]:
# Z-score scatter: FP vs non-FP cases
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. z_dcor vs z_eta2 (most independent pair)
ax = axes[0]
non_fp = fp_df[~fp_df['is_fp']]
fp_only = fp_df[fp_df['is_fp']]
ax.scatter(non_fp['z_dcor'], non_fp['z_eta2'], s=3, alpha=0.2, c='steelblue', label='True Negative')
ax.scatter(fp_only['z_dcor'], fp_only['z_eta2'], s=20, alpha=0.8, c='red', marker='x', label='False Positive')
ax.set_xlabel('z_dcor'); ax.set_ylabel('z_eta2')
ax.set_title('z_dcor vs z_eta2')
ax.legend(fontsize=8)

# 2. z_pearson vs z_spearman
ax = axes[1]
ax.scatter(non_fp['z_pearson'], non_fp['z_spearman'], s=3, alpha=0.2, c='steelblue')
ax.scatter(fp_only['z_pearson'], fp_only['z_spearman'], s=20, alpha=0.8, c='red', marker='x')
ax.set_xlabel('z_pearson'); ax.set_ylabel('z_spearman')
ax.set_title('z_pearson vs z_spearman')

# 3. FP driver distribution
ax = axes[2]
if len(fp_only) > 0:
    driver_counts = fp_only['driver'].value_counts()
    ax.bar(driver_counts.index, driver_counts.values, color='salmon')
    ax.set_ylabel('Count')
    ax.set_title(f'FP Driver Metric (n={len(fp_only)})')
else:
    ax.text(0.5, 0.5, 'No FP cases', ha='center', va='center', transform=ax.transAxes)

plt.suptitle(f'False Positive Analysis — Combined True Null (n={n_null_total})', fontsize=13)
plt.tight_layout()
fig.savefig(VIZ_DIR / '6_fp_analysis.png', bbox_inches='tight')
plt.show()

/var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/ipykernel_67582/2900943741.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# FP rate by x_distribution
fig, ax = plt.subplots(figsize=(10, 5))
xdists = sorted(fp_df['x_distribution'].unique())
fp_rates = [fp_df[fp_df['x_distribution']==xd]['is_fp'].mean() for xd in xdists]
colors = ['salmon' if r > 0.05 else 'steelblue' for r in fp_rates]
ax.barh(range(len(xdists)), fp_rates, color=colors, alpha=0.7)
ax.axvline(0.05, color='red', ls='--', lw=1.5, label='α=0.05')
ax.set_yticks(range(len(xdists)))
ax.set_yticklabels(xdists)
ax.set_xlabel('FP rate')
ax.set_title(f'FP Rate by x_distribution (Combined True Null, n={n_null_total})')
ax.legend()

for i, (xd, rate) in enumerate(zip(xdists, fp_rates)):
    n = (fp_df['x_distribution']==xd).sum()
    fp_n = fp_df[fp_df['x_distribution']==xd]['is_fp'].sum()
    ax.text(rate + 0.002, i, f'{fp_n}/{n}', va='center', fontsize=8)

plt.tight_layout()
fig.savefig(VIZ_DIR / '7_fp_rate_by_xdist.png', bbox_inches='tight')
plt.show()

/var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/ipykernel_67582/12283822.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. p-value Calibration Under True Null

If the test is correctly calibrated, p-values under True Null should be Uniform(0,1).

In [16]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. p-value histogram
ax = axes[0]
ax.hist(fp_df['p_value'], bins=50, color='steelblue', edgecolor='white', density=True)
ax.axhline(1.0, color='red', ls='--', lw=1.5, label='Uniform(0,1)')
ax.set_xlabel('p-value'); ax.set_ylabel('Density')
ax.set_title(f'p-value Distribution (n={n_null_total})')
ax.legend()

# 2. p-value CDF vs uniform
ax = axes[1]
p_sorted = np.sort(fp_df['p_value'].values)
ecdf = np.arange(1, len(p_sorted)+1) / len(p_sorted)
ax.plot(p_sorted, ecdf, 'b-', lw=1.5, label='Empirical CDF')
ax.plot([0, 1], [0, 1], 'r--', lw=1, label='Uniform')
ax.set_xlabel('p-value'); ax.set_ylabel('Cumulative fraction')
ax.set_title('p-value CDF vs Uniform')
ax.legend()

# 3. QQ-plot of p-values
ax = axes[2]
theoretical = np.linspace(0, 1, n_null_total + 2)[1:-1]
ax.scatter(theoretical, p_sorted, s=2, alpha=0.5, c='steelblue')
ax.plot([0, 1], [0, 1], 'r--', lw=1)
ax.set_xlabel('Theoretical (Uniform)'); ax.set_ylabel('Observed p-value')
ax.set_title('QQ-plot')

plt.suptitle(f'p-value Calibration Under True Null (S3+S4, n={n_null_total})', fontsize=13)
plt.tight_layout()
fig.savefig(VIZ_DIR / '8_pvalue_calibration.png', bbox_inches='tight')
plt.show()

/var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/ipykernel_67582/2494786714.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Effect Size by SNR (S3 Signal Cases)

In [17]:
# Effect size: obs / null_median ratio across SNR (S3 signal cases)
signal_idx = np.where(s3_signal_mask)[0]
snr_vals = pd.to_numeric(cases.loc[s3_signal_mask, 'snr'], errors='coerce').values

s3_metric_info = [
    ('|Pearson|',  s3_pearson_obs,  s3_pearson_null),
    ('|Spearman|', s3_spearman_obs, s3_spearman_null),
    ('dcor',       s3_dcor_obs,     s3_dcor_null),
    ('η²',        s3_eta2_obs,     s3_eta2_null),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
fig.suptitle('Effect Size: obs / null_median by SNR (Signal cases)', fontsize=13)
snr_unique = np.sort(np.unique(snr_vals[~np.isnan(snr_vals)]))

for col, (mname, obs_arr, null_arr) in enumerate(s3_metric_info):
    ratios_by_snr = []
    for snr in snr_unique:
        mask = snr_vals == snr
        idxs = signal_idx[mask]
        if len(idxs) == 0: continue
        obs = obs_arr[idxs]
        null_med = np.median(null_arr[idxs].astype(np.float64), axis=1)
        ratio = obs / np.clip(null_med, 1e-10, None)
        ratios_by_snr.append((snr, np.median(ratio), np.percentile(ratio, 25), np.percentile(ratio, 75)))
    rs = np.array(ratios_by_snr)
    ax = axes[col]
    ax.plot(range(len(rs)), rs[:, 1], 'o-', color='steelblue', lw=2)
    ax.fill_between(range(len(rs)), rs[:, 2], rs[:, 3], alpha=0.2, color='steelblue')
    ax.set_xticks(range(len(rs)))
    ax.set_xticklabels([f'{s:.1f}' if s < 100 else ('inf' if np.isinf(s) else f'{s:.0f}') for s in rs[:, 0]],
                       rotation=45, fontsize=8)
    ax.set_xlabel('SNR'); ax.set_ylabel('obs / null_median')
    ax.set_title(mname)
    ax.axhline(1, color='k', ls='--', lw=0.5)
    ax.set_yscale('log')

plt.tight_layout()
fig.savefig(VIZ_DIR / '9_effect_size_by_snr.png', bbox_inches='tight')
plt.show()

/var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/ipykernel_67582/1031892368.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Cross-Metric Null Correlation

In [18]:
# Cross-metric null correlation (sample from S3)
sample_corr_idx = rng.choice(n_s3, size=500, replace=False)

corr_pairs = [('pearson-spearman', s3_pearson_null, s3_spearman_null),
              ('pearson-dcor',     s3_pearson_null, s3_dcor_null),
              ('pearson-eta2',     s3_pearson_null, s3_eta2_null),
              ('spearman-dcor',    s3_spearman_null, s3_dcor_null),
              ('spearman-eta2',    s3_spearman_null, s3_eta2_null),
              ('dcor-eta2',        s3_dcor_null, s3_eta2_null)]

null_corr_records = []
for idx in sample_corr_idx:
    for pair_name, arr_a, arr_b in corr_pairs:
        a = arr_a[idx].astype(np.float64)
        b = arr_b[idx].astype(np.float64)
        r, _ = stats.pearsonr(a, b)
        null_corr_records.append({'idx': idx, 'pair': pair_name, 'r': r})

null_corr_df = pd.DataFrame(null_corr_records)
print('Cross-metric null correlation (500 sampled S3 cases):')
print(null_corr_df.groupby('pair')['r'].describe().round(3).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
for pair_name in [p[0] for p in corr_pairs]:
    sub = null_corr_df[null_corr_df['pair'] == pair_name]
    ax.hist(sub['r'], bins=40, alpha=0.5, label=pair_name, density=True)
ax.set_xlabel('Pearson r between null distributions')
ax.set_title('Cross-Metric Null Correlation', fontsize=12)
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(VIZ_DIR / '10_null_cross_correlation.png', bbox_inches='tight')
plt.show()

Cross-metric null correlation (500 sampled S3 cases):
                  count   mean    std    min    25%    50%    75%    max
pair                                                                    
dcor-eta2         500.0  0.437  0.057  0.204  0.404  0.440  0.477  0.615
pearson-dcor      500.0  0.791  0.069  0.510  0.751  0.797  0.835  0.939
pearson-eta2      500.0  0.310  0.048  0.184  0.277  0.310  0.341  0.476
pearson-spearman  500.0  0.816  0.153  0.096  0.776  0.861  0.911  0.987
spearman-dcor     500.0  0.766  0.089  0.194  0.757  0.789  0.817  0.896
spearman-eta2     500.0  0.264  0.070 -0.012  0.231  0.273  0.309  0.433


/var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/ipykernel_67582/4031326128.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Summary

In [ ]:
# Summary table
summary_rows = []
for mname, obs_arr, null_arr in [('|Pearson|', s3_pearson_obs, s3_pearson_null),
                                  ('|Spearman|', s3_spearman_obs, s3_spearman_null),
                                  ('dcor', s3_dcor_obs, s3_dcor_null),
                                  ('η²', s3_eta2_obs, s3_eta2_null)]:
    null_f64 = null_arr.astype(np.float64)
    meds = np.median(null_f64, axis=1)
    iqrs = np.percentile(null_f64, 75, axis=1) - np.percentile(null_f64, 25, axis=1)
    summary_rows.append({
        'metric': mname,
        'obs_mean': obs_arr.mean(), 'obs_std': obs_arr.std(),
        'null_med_mean': meds.mean(), 'null_med_std': meds.std(),
        'null_iqr_mean': iqrs.mean(), 'null_iqr_std': iqrs.std(),
    })

summary_df = pd.DataFrame(summary_rows)
print('=== Null Distribution Summary (S3, all 114K cases) ===')
print(summary_df.to_string(index=False, float_format='%.4f'))
print()
print(f'=== FP Calibration (Combined S3+S4 True Null, {n_null_total}) ===')
fp_rate = fp_df['is_fp'].mean()
print(f'FP rate: {fp_df["is_fp"].sum()}/{n_null_total} = {fp_rate:.2%}')
print(f'FP by x_distribution:')
for xd in sorted(fp_df['x_distribution'].unique()):
    sub = fp_df[fp_df['x_distribution'] == xd]
    print(f'  {xd:>15s}: {sub["is_fp"].sum():>3d}/{len(sub)} = {sub["is_fp"].mean():.1%}')

print(f'\n=== Variance-Only Detection (S4, {n_var} cases) ===')
for _, r in var_det_df[~var_det_df['metric'].str.startswith('  ')].iterrows():
    print(f'  {r["metric"]:>12s}: {r["detection_rate"]:.1%}')

## 11. Variance-Only Detection Analysis

Cases where E[Y|X] = 0 but Var[Y|X] depends on X (non-constant spread).
Which core metrics can detect this relationship?

In [ ]:
# Per-metric detection rate on Variance-only cases
var_metric_info = [
    ('|Pearson|',  var_pearson_obs,  var_pearson_null),
    ('|Spearman|', var_spearman_obs, var_spearman_null),
    ('dcor',       var_dcor_obs,     var_dcor_null),
    ('η²',        var_eta2_obs,     var_eta2_null),
    ('dcov',       var_dcov_obs,     var_dcov_null),
]

var_det_rows = []
for mname, obs_arr, null_arr in var_metric_info:
    p_vals = np.array([
        (np.sum(null_arr[i].astype(np.float64) >= obs_arr[i]) + 1) / (n_perm + 1)
        for i in range(n_var)
    ])
    det_rate = (p_vals <= 0.05).mean()
    var_det_rows.append({'metric': mname, 'detection_rate': det_rate, 'n': n_var})

    # By spread_pattern
    for sp in ['increasing', 'decreasing', 'middle_high']:
        sp_mask = var_meta['spread_pattern'] == sp
        sp_p = p_vals[sp_mask.values]
        var_det_rows.append({
            'metric': f'  {mname} ({sp})',
            'detection_rate': (sp_p <= 0.05).mean(),
            'n': len(sp_p),
        })

var_det_df = pd.DataFrame(var_det_rows)

print(f'═══ Variance-Only Detection (n={n_var}, α=0.05) ═══')
print(f'{"Metric":>30s}  {"Det rate":>9s}  {"n":>6s}')
print('-' * 52)
for _, r in var_det_df.iterrows():
    print(f'{r["metric"]:>30s}  {r["detection_rate"]:>8.1%}  {int(r["n"]):>6d}')

In [ ]:
# Visualize: Variance-only null histograms for dcor (should detect) vs pearson (should not)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Variance-Only: Null Distributions — dcor (top) vs |Pearson| (bottom)', fontsize=13)

for col, sp in enumerate(['increasing', 'decreasing', 'middle_high']):
    sp_mask = var_meta['spread_pattern'] == sp
    sp_idx = np.where(sp_mask.values)[0]
    if len(sp_idx) == 0:
        continue
    i = sp_idx[0]

    # dcor
    ax = axes[0, col]
    nv = var_dcor_null[i].astype(np.float64)
    ov = float(var_dcor_obs[i])
    ax.hist(nv, bins=40, color='#2ca02c', alpha=0.7, density=True)
    ax.axvline(ov, color='red', lw=2, label=f'obs={ov:.3f}')
    p = (np.sum(nv >= ov) + 1) / (n_perm + 1)
    ax.set_title(f'dcor — {sp}  (p={p:.3f})', fontsize=10)
    ax.legend(fontsize=7)

    # pearson
    ax = axes[1, col]
    nv = var_pearson_null[i].astype(np.float64)
    ov = float(var_pearson_obs[i])
    ax.hist(nv, bins=40, color='#1f77b4', alpha=0.7, density=True)
    ax.axvline(ov, color='red', lw=2, label=f'obs={ov:.3f}')
    p = (np.sum(nv >= ov) + 1) / (n_perm + 1)
    ax.set_title(f'|Pearson| — {sp}  (p={p:.3f})', fontsize=10)
    ax.legend(fontsize=7)

plt.tight_layout()
fig.savefig(VIZ_DIR / '11_variance_only_detection.png', bbox_inches='tight')
plt.show()